# Facial-Expression CNN for the Interview Simulator

**What this trains:** a CNN that classifies a face crop into 7 *expressions*
(angry, disgust, fear, happy, neutral, sad, surprise) using the FER-2013 dataset
and MobileNetV2 transfer learning, then exports it to **TensorFlow.js** so it runs
**in the browser** (the face never leaves the user's device).

**Read this before you pitch it:** the model outputs *facial expressions*, not
*emotions*. A "fear/tense" expression is **not** a measurement of anxiety. In the
app, feed these probabilities into a clearly-labelled **tension proxy** — never a
raw "anxiety: 0.7". To honestly estimate anxiety you must collect self-reported
labels and train a separate fusion model on them.

**Expected accuracy:** ~60–70% on FER-2013. That is roughly state-of-the-art for
this noisy dataset (human agreement is ~65%), so don't promise more.

Runtime: **Runtime -> Change runtime type -> GPU**.

In [ ]:
# 1. Install the TensorFlow.js converter (TF + Keras are already in Colab)
!pip -q install tensorflowjs
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 2. Get FER-2013

Easiest source is the Kaggle dataset `msambare/fer2013` (image folders:
`train/<emotion>/*.jpg` and `test/<emotion>/*.jpg`).

1. On kaggle.com: Account -> Create New API Token -> downloads `kaggle.json`.
2. Run the next cell and upload that `kaggle.json` when prompted.

In [ ]:
# Upload kaggle.json, then download + unzip FER-2013
from google.colab import files
import os, zipfile

if not os.path.exists('data/train'):
    print("Upload your kaggle.json:")
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(list(up.values())[0])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !kaggle datasets download -d msambare/fer2013 -p data --unzip
print("train dir contents:", os.listdir('data/train'))

In [ ]:
# 3. Build tf.data pipelines
import tensorflow as tf

IMG = 96          # MobileNetV2 wants >= 96px, 3 channels
BATCH = 64
TRAIN_DIR, TEST_DIR = 'data/train', 'data/test'

raw_train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, color_mode='grayscale', image_size=(48, 48),
    batch_size=BATCH, label_mode='int', shuffle=True, seed=42)
raw_val = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, color_mode='grayscale', image_size=(48, 48),
    batch_size=BATCH, label_mode='int', shuffle=False)

CLASS_NAMES = raw_train.class_names           # alphabetical order — keep for the browser!
print("CLASS ORDER (use this exact index order in JS):", CLASS_NAMES)

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def prep(img, label):
    img = tf.image.resize(img, (IMG, IMG))
    img = tf.image.grayscale_to_rgb(img)      # 1ch -> 3ch
    img = preprocess_input(img)               # MobileNetV2 scaling
    return img, label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train.map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = raw_val.map(prep,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
# 4. Class weights (FER is imbalanced — 'disgust' is tiny)
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

labels = np.concatenate([y.numpy() for _, y in raw_train], axis=0)
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=labels)
class_weight = {i: float(w) for i, w in enumerate(weights)}
print("class weights:", class_weight)

In [ ]:
# 5. Model — MobileNetV2 backbone (frozen) + small classification head
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=(IMG, IMG, 3))
x = layers.RandomFlip('horizontal')(inputs)   # light augmentation
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 6. Train the head
cbs = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, monitor='val_loss'),
]
hist = model.fit(train_ds, validation_data=val_ds, epochs=20,
                 class_weight=class_weight, callbacks=cbs)

In [ ]:
# 7. Fine-tune the top of the backbone at a low learning rate (optional, helps)
base.trainable = True
for layer in base.layers[:-30]:      # keep early layers frozen
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=8,
                  class_weight=class_weight, callbacks=cbs)

In [ ]:
# 8. Evaluate + confusion matrix
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
y_prob = model.predict(val_ds)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45)
plt.yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix'); plt.colorbar()
plt.show()

### Quick bias check (do not skip)

FER-2013 is skewed in lighting and demographics. Before trusting this in the app,
test it on a few of **your own** webcam frames across different skin tones and
lighting. If accuracy collapses for some groups, a "tension" score built on it is
really a lighting/skin score — unfair and a credibility risk. Note what you find.

In [ ]:
# 9. Export to TensorFlow.js for in-browser inference
import tensorflowjs as tfjs, json, shutil
tfjs.converters.save_keras_model(model, 'tfjs_model')

# save the class order so the browser maps output indices correctly
with open('tfjs_model/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)

shutil.make_archive('fer_tfjs_model', 'zip', 'tfjs_model')
from google.colab import files
files.download('fer_tfjs_model.zip')
print("Done. Unzip into your web app and load with tf.loadLayersModel('.../model.json').")

## 10. Using it in the browser (wiring sketch)

```html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
```

```js
const fer = await tf.loadLayersModel('models/fer_tfjs/model.json');
const CLASSES = ['angry','disgust','fear','happy','neutral','sad','surprise']; // from class_names.json

// faceCanvas = a 96x96 crop of the face (use the MediaPipe bbox to crop the video)
function expressionProbs(faceCanvas) {
  return tf.tidy(() => {
    let t = tf.browser.fromPixels(faceCanvas).resizeBilinear([96,96]).toFloat();
    t = t.div(127.5).sub(1).expandDims(0);          // MobileNetV2 preprocessing
    return fer.predict(t).dataSync();                // 7 probabilities
  });
}
```

Per answer, average the `fear + sad` probabilities and the frame-to-frame
*variance* of the predictions; feed those (normalized) into `presence.tensionProxy`
in `scoring.js`. Label the result **"Composure (proxy)"** in the UI.